In [1]:
# Tested with transformers==4.46.3, trl==0.12.1, datasets==3.1.0, bitsandbytes==0.45.0, peft==0.13.2, accelerate==1.1.1 in the finetuning notebook from huggingface https://huggingface.co/learn/cookbook/en/fine_tuning_smol_vlm_sft_trl#2-load-dataset-

In [2]:
! nvidia-smi

Fri Mar 14 15:58:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.22                 Driver Version: 552.22         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:16:00.0 Off |                  Off |
| 30%   33C    P8             15W /  300W |    9093MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from datasets import load_dataset
import torch
from transformers import Idefics3ForConditionalGeneration, AutoProcessor
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image

In [4]:
from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'YOUR_HF_TOKEN'  #Write token for SMOLVLM-500M Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

Logged in as aadhibest


In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [6]:
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [7]:
def load_image(image_path, target_size=(512, 512)):
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [8]:
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

Training Dataset Size: 3585
Evaluation Dataset Size: 897


In [9]:
train_dataset[0]

{'id': '3950',
 'image': 'circuit_imgs_03/auto_contrast/P2P_R10_L36_CHARIMP50_PROP0.2_TDR_ANALYSIS_auto_contrast.png',
 'conversations': [{'from': 'human', 'value': 'What is shown in this graph?'},
  {'from': 'gpt',
   'value': "The Time Domain Reflectometry (TDR) analysis graph illustrates the voltage, current, and impedance characteristics of a transmission line featuring a characteristic impedance of 50Ω, a propagation factor of 0.2, and a P2P topology circuit that includes a 10Ω resistor and a 36nH inductor. The top panel's voltage response indicates an initial rapid increase from 0V to about 250mV within the first 0.5ns, signifying a quick signal transition. This voltage remains stable until there is a slight rise around 2.8ns, which may be attributed to impedance mismatch at a discontinuity. The middle panel depicts the current waveform, showing a swift increase to approximately 5mA and maintaining stability until a minor dip at 2.8ns, confirming a reflection due to impedance var

In [10]:
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [11]:
train_dataset_formatted[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.\n\nFor circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.\n\nFocus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy.'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.Image.Image image mode=RGB size=512x512>},
   {'type': 'text', 'text': '

In [12]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_smolvlm_500m"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_smolvlm_500m"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/3585 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/897 [00:00<?, ? examples/s]

Train dataset saved to: ./datasets/train_dataset_13_03_2025_smolvlm_500m
Eval dataset saved to: ./datasets/eval_dataset_13_03_2025_smolvlm_500m


In [13]:
#Remove Model and Clean GPU

def clear_memory():
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


Fine Tune Model

In [14]:
model_id = "HuggingFaceTB/SmolVLM-500M-Instruct"

In [15]:
# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load model and tokenizer
model = Idefics3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
processor = AutoProcessor.from_pretrained(model_id)

In [16]:
# Configure LoRA
peft_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    use_dora=True,
    init_lora_weights="gaussian"
)

# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 5,088,256 || all params: 512,570,560 || trainable%: 0.9927


In [17]:
# Configure training arguments using SFTConfig
training_args = SFTConfig(
    output_dir="smolvlm-500m-instruct-13-03-2025",
    logging_dir="smolvlm-500m-instruct-13-03-2025/logs",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=10,
    optim="adamw_torch_fused",
    bf16=True,
    push_to_hub=True,
    report_to="tensorboard",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    #gradient_checkpointing_kwargs={"use_reentrant": False}, #Enable this when using DDP as device_map
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [18]:
image_token_id = processor.tokenizer.additional_special_tokens_ids[
            processor.tokenizer.additional_special_tokens.index("<image>")]

def collate_fn(examples):
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

    image_inputs = []
    for example in examples:
      image = example[1]['content'][0]['image']
      if image.mode != 'RGB':
          image = image.convert('RGB')
      image_inputs.append([image])

    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)
    
    # Move all tensors to the same device as the model
    #batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100  # Mask padding tokens in labels
    labels[labels == image_token_id] = -100  # Mask image token IDs in labels

    batch["labels"] = labels

    return batch

In [19]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

C:\Users\hal-pg2425-5\AppData\Local\Temp\8\ipykernel_38708\2552207196.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [20]:
%load_ext tensorboard
%tensorboard --logdir smolvlm-500m-instruct-13-03-2025/logs --port=6012

In [21]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,2.704000
20,2.651000
30,2.583900
40,2.496700
50,2.357600
60,2.231800
70,2.126800
80,2.008500
90,1.835300
100,1.673500


TrainOutput(global_step=224, training_loss=1.6875016795737403, metrics={'train_runtime': 7070.6368, 'train_samples_per_second': 0.507, 'train_steps_per_second': 0.032, 'total_flos': 1.657140212027904e+16, 'train_loss': 1.6875016795737403})

In [22]:
trainer.save_model(training_args.output_dir)

In [23]:
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aadhibest/smolvlm-500m-instruct-13-03-2025/commit/a8bde2f81304bde741d65f3260099bae33e6b30f', commit_message='End of training', commit_description='', oid='a8bde2f81304bde741d65f3260099bae33e6b30f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhibest/smolvlm-500m-instruct-13-03-2025', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhibest/smolvlm-500m-instruct-13-03-2025'), pr_revision=None, pr_num=None)

In [24]:
clear_memory()

GPU allocated memory: 0.02 GB
GPU reserved memory: 0.12 GB
